# Stage 1 — full-ROI proposal detector
**CUDA: required. Training only.**

The network sees the complete 1024×1024 ROI and predicts class-agnostic nucleus proposals on a stride-2 grid with learned subpixel offsets. Each fold resumes from `latest.pt` only when its stored training signature matches the current critical settings. The fixed final-epoch EMA checkpoint is used for OOF proposal generation.


In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("GPU Name:", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import sys

IN_COLAB = Path("/content").is_dir()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research/PUMA")
else:
    PROJECT_ROOT = Path("/media/ibmelab/ibme31/YUD/PUMA")

CODE_ROOT = PROJECT_ROOT / "PUMA_Nuclei_Pipeline"
if not CODE_ROOT.is_dir():
    raise FileNotFoundError(f"Code folder not found: {CODE_ROOT}")
sys.path.insert(0, str(CODE_ROOT / "src"))

from puma_nuclei.config import PipelineConfig

print("Environment :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)


## Editable Stage-1 configuration — 24 GB defaults
The defaults are intentionally conservative. If memory is tight, reduce micro-batch size only; keep the effective batch via gradient accumulation.


In [ ]:
config = PipelineConfig()
config.paths.project_root = str(PROJECT_ROOT)
config.data.number_of_folds = 5
config.data.seed = 42
config.data.cpu_workers = -1

# Optional Colab optimization: train from local SSD instead of Google Drive memmap I/O.
# Checkpoints/results still go to PROJECT_ROOT/PUMA_stage1_training_outputs.
USE_LOCAL_PREPROCESSED_CACHE = bool(IN_COLAB)
if USE_LOCAL_PREPROCESSED_CACHE:
    import shutil
    source = PROJECT_ROOT / "PUMA_preprocessed"
    local = Path("/content/PUMA_preprocessed")
    required = ["images.npy", "stage1_heatmaps.npy", "stage1_point_targets.npy", "folds.npy"]
    if not source.is_dir():
        raise FileNotFoundError(f"Shared preprocessing not found: {source}")
    def _same_cached_file(name):
        src, dst = source / name, local / name
        if not src.is_file() or not dst.is_file():
            return False
        a, b = src.stat(), dst.stat()
        return a.st_size == b.st_size and a.st_mtime_ns == b.st_mtime_ns
    summary_current = (
        (source / "preprocessing_summary.json").is_file()
        and (local / "preprocessing_summary.json").is_file()
        and (source / "preprocessing_summary.json").read_bytes()
            == (local / "preprocessing_summary.json").read_bytes()
    )
    if not summary_current or not all(_same_cached_file(name) for name in required):
        print(f"Refreshing shared preprocessing on local SSD: {local}")
        shutil.rmtree(local, ignore_errors=True)
        shutil.copytree(source, local, copy_function=shutil.copy2)
    config.paths.shared_preprocessing_directory = str(local)

# 24-GB Stage-1 profile: physical batch 8, gradient accumulation x2 => effective 16.
config.stage1.epochs = 150
config.stage1.base_channels = 48
config.stage1.fpn_channels = 96
config.stage1.micro_batch_size = 4
config.stage1.effective_batch_size = 8
config.stage1.learning_rate = 3.5e-4
config.stage1.backbone_learning_rate_scale = 1.0
config.stage1.minimum_learning_rate = 2.0e-6
config.stage1.weight_decay = 1.0e-4
config.stage1.warmup_epochs = 5
config.stage1.gradient_clip_norm = 1.0
config.stage1.ema_decay = 0.9995

# Confidence/FP optimization.
config.stage1.heatmap_loss_weight = 0.55
config.stage1.offset_loss_weight = 0.18
config.stage1.quality_loss_weight = 0.12
config.stage1.uncertainty_loss_weight = 0.05
config.stage1.hard_negative_loss_weight = 0.10
config.stage1.hard_negative_topk_per_image = 256
config.stage1.hard_negative_start_epoch = 5
config.stage1.hard_negative_ramp_epochs = 10

# Histology-safe D4 + mild stain/photometric augmentation.
config.stage1.use_d4_augmentation = True
config.stage1.d4_augmentation_probability = 0.875
config.stage1.photometric_augmentation = True
config.stage1.offset_range_grid_units = 2.5
config.stage1.quality_target_sigma_grid_units = 0.75

# Fused proposal score + recall-constrained threshold/NMS search.
config.stage1.proposal_quality_power = 1.0
config.stage1.proposal_uncertainty_penalty = 0.0
config.stage1.validation_thresholds = (
    0.005, 0.010, 0.020, 0.030, 0.040, 0.050, 0.075, 0.100,
    0.125, 0.150, 0.200, 0.250, 0.300, 0.400, 0.500, 0.650, 0.800, 0.900, 0.950,
)
config.stage1.validation_suppression_radii_pixels = (3.0, 4.0, 5.0, 6.0, 8.0)
config.stage1.validation_min_recall = 0.93
config.stage1.local_max_radius = 2
config.stage1.suppression_radius_pixels = 3.0
config.stage1.max_candidates_per_roi = 4000
config.stage1.checkpoint_every_epochs = 1

#config.runtime.use_bfloat16 = True
config.runtime.use_bfloat16 = False
config.runtime.allow_tf32 = True
config.runtime.channels_last = True
config.runtime.fused_adamw = True
config.runtime.pin_memory = True
config.runtime.persistent_workers = True
config.validate()

print("Stage1 output:", config.paths.stage1_outputs)
print("Shared preprocessing:", config.paths.preprocessed)
print("micro/effective batch:", config.stage1.micro_batch_size, config.stage1.effective_batch_size)
print("accumulation steps:", config.stage1.accumulation_steps)
print("validation recall floor:", config.stage1.validation_min_recall)


In [ ]:
import json
import matplotlib.pyplot as plt

PLOT_EVERY_EPOCHS = 5

def _read_jsonl(path):
    if not path.is_file():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows

def _stage1_metric(record, metric):
    if metric == "loss":
        return float(record["train_loss"])
    return float(record["binary_detection"][metric])

def plot_stage1_training(fold, record, run_directory):
    completed_epoch = int(record["epoch"]) + 1
    if completed_epoch % PLOT_EVERY_EPOCHS != 0 and completed_epoch != int(config.stage1.epochs):
        return

    run_directory = Path(run_directory)
    plot_directory = run_directory / "training_plots"
    plot_directory.mkdir(parents=True, exist_ok=True)
    metric_specs = [
        ("f1", "F1@15px", "F1"),
        ("recall", "Recall@15px", "Recall"),
        ("precision", "Precision@15px", "Precision"),
        ("accuracy", "Accuracy@15px", "Accuracy"),
        ("loss", "Training Loss", "Loss"),
    ]

    for metric_key, title, ylabel in metric_specs:
        fig, ax = plt.subplots(figsize=(9, 5))
        plotted = 0
        for fold_id in range(int(config.data.number_of_folds)):
            history_path = run_directory / f"fold_{fold_id}" / "history.jsonl"
            history = _read_jsonl(history_path)
            if not history:
                continue
            epochs = [int(row["epoch"]) + 1 for row in history]
            values = [_stage1_metric(row, metric_key) for row in history]
            ax.plot(epochs, values, marker="o", markersize=3, linewidth=1.6, label=f"Fold {fold_id}")
            plotted += 1
        ax.set_title(f"Stage 1 - {title}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
        if plotted:
            ax.legend()
        fig.tight_layout()
        fig.savefig(plot_directory / f"{metric_key}.png", dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)



## Qualitative train/validation visualization
The notebook displays **two sharp 4×4 dashboards: 16 train + 16 validation ROIs** for a selected fold, doubling the previous qualitative sample budget. This is visualization only and does not change augmentation, training, history, checkpoints, or fold routing.


In [ ]:
from puma_nuclei.utils.visualization import visualize_stage1_train_validation

TRAIN_VISUALIZATION_IMAGES = 16
VALIDATION_VISUALIZATION_IMAGES = 16
VISUALIZATION_FOLD = 0
VISUALIZATION_COLUMNS = 4
VISUALIZATION_CELL_SIZE_INCHES = 4.0
VISUALIZATION_DPI = 300

visualize_stage1_train_validation(
    config,
    fold=VISUALIZATION_FOLD,
    train_count=TRAIN_VISUALIZATION_IMAGES,
    validation_count=VALIDATION_VISUALIZATION_IMAGES,
    max_columns=VISUALIZATION_COLUMNS,
    dashboard_dpi=VISUALIZATION_DPI,
    cell_size_inches=VISUALIZATION_CELL_SIZE_INCHES,
)


In [ ]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())

print("\nGPU:", torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"VRAM free : {free / 1024**3:.2f} GB")
print(f"VRAM total: {total / 1024**3:.2f} GB")

print("\nBF16 supported:", torch.cuda.is_bf16_supported())
print("cuDNN enabled:", torch.backends.cudnn.enabled)
print("cuDNN benchmark:", torch.backends.cudnn.benchmark)
print("cuDNN deterministic:", torch.backends.cudnn.deterministic)

print("\nStage1 micro:", config.stage1.micro_batch_size)
print("Stage1 effective:", config.stage1.effective_batch_size)

print("channels_last:", config.runtime.channels_last)
print("use_bfloat16:", config.runtime.use_bfloat16)
print("deterministic:", config.runtime.deterministic)
print("allow_tf32:", config.runtime.allow_tf32)

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA is required for Stage-1 training.'

from puma_nuclei.stage1.trainer import train_all_stage1_folds
from puma_nuclei.stage1.registry import stage1_run_directory

# Train normally — NO visualization during training
stage1_results = train_all_stage1_folds(
    config,
    #folds=[0],
    epoch_callback=None
)

# Plot ONCE, after every fold has completed
plot_stage1_training(
    fold=0,  # unused by the plotting function
    record={"epoch": config.stage1.epochs - 1},
    run_directory=stage1_run_directory(config),
)

stage1_results

Every epoch appends `history.jsonl` and `history.csv` and prints the class-agnostic binary detection F1, accuracy, precision, and recall at 15 px. Stage 1 is a proposal detector only; nuclei classes are learned in Stage 2. Held-fold performance is diagnostic only, and the fixed final epoch/EMA checkpoint is used for OOF proposal generation.


For point detection there is no meaningful true-negative count, so `Accuracy@15px` is defined as `TP / (TP + FP + FN)` (set/detection accuracy).


In [ ]:
if IN_COLAB:
    from google.colab import runtime
    print("Notebook complete. Disconnecting Colab runtime...")
    runtime.unassign()
